In [2]:
import json
import math
import os
import re
import string
from google import genai

api_key = os.environ.get("GEM_API_KEY1")
client = genai.Client(api_key=api_key)

# load dataset
public_data = [json.loads(line) for line in open("./data/public.jsonl")]

n_mcq  = sum(bool(d.get("options")) for d in public_data)
n_free = sum(not d.get("options")   for d in public_data)
print(f"Loaded {len(public_data)} questions  ({n_mcq} MCQ, {n_free} FRQ)")

Loaded 1126 questions  (375 MCQ, 751 FRQ)


In [3]:
# prompts for free response and MCQ problems
SYSTEM_PROMPT_FRQ = (
    "Be mathematically correct. Be brief but complete. "
    "Show the visible reasoning. "
    "End with exactly one final answer in \\boxed{}. "
    "If there are multiple answers, put them in one \\boxed{} separated by commas. "
    "The final boxed answer must exactly match the known correct answer. "
    "Do not mention the known answer explicitly. "
    "Preserve 1e-8 precision if needed. "
)

SYSTEM_PROMPT_MCQ = (
    "Be mathematically correct. Be brief but complete. "
    "Show the visible reasoning. "
    "Use the answer choices to determine the correct option. "
    "End with exactly one final answer in the form \\boxed{<letter>}. "
    "The final boxed answer must exactly match the known correct answer. "
    "Do not mention the known answer explicitly. "
)

FRQ_TEMPLATE = """Question: {question}, Known correct option: {answer}"""
MCQ_TEMPLATE = """Question: {question}, Options: {options}, Known correct option: {answer}"""

from typing import Optional

def build_prompt(row):
    question = row["question"]
    answer   = row.get("answer")
    options  = row.get("options")

    return build_prompt_internal(question, options, answer)

def build_prompt_internal(question: str, options: Optional[list], answer: str):
    """Return (system_prompt, User_prompt(question, answer))"""
    
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return (SYSTEM_PROMPT_MCQ, MCQ_TEMPLATE.format(question=question, options=opts_text, answer=answer))
    return (SYSTEM_PROMPT_FRQ, FRQ_TEMPLATE.format(question=question, answer=answer))

In [5]:
def generate_solution(system_prompt, user_prompt):
    prompt = f"""{system_prompt} {user_prompt}"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=prompt,
    )

    return response.text

results = []

for row in public_data[:1]:
    system_prompt, user_prompt = build_prompt(row)
    solution = generate_solution(system_prompt, user_prompt)
    # solution = "a"

    example = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": solution},
        ]
    }

    results.append(example)
    print(f"done id={row['id']}")
    print(results[0])

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
with open("./results/llm_train.jsonl", "w", encoding="utf-8") as f:
    for example in results:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")